# Exponential Decay Characterization of d' vs ISI Curves

Analyse the vectorized (σ₀, σ, η) parameter sweep by fitting an exponential
decay model to each triple's d' vs ISI curve:

$$d'(\text{ISI}) = A \, e^{-\lambda \, \text{ISI}} + C$$

where **A** is the decaying amplitude, **λ** the decay rate, and **C** the
asymptotic floor.

**Approach — fix two, vary one:**  for each hyperparameter, fix the other two
and sweep the target parameter across its grid.  Each setting yields a d' vs ISI
curve that is fit by the exponential above, mapping each (σ₀, σ, η) triple to
decay-function parameters (A, λ, C).  We then ask: *how does each
hyperparameter control the shape of memory decay?*

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy.optimize import curve_fit
from scipy.stats import spearmanr
from matplotlib.colors import Normalize, LogNorm
from matplotlib import cm

matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── Point this to your vectorized grid-search output directory ──
RESULTS_DIR = '/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/reports/figures/2d_grid_search_vectorized_dense'

# Load master grid arrays
grid = np.load(os.path.join(RESULTS_DIR, 'grid_search_results_vec.npz'))
sigma0_grid = grid['sigma0_grid']
sigma_grid  = grid['sigma_grid']
eta_grid    = grid['eta_grid']
ISI_VALUES  = tuple(grid['isi_values'].astype(int))
isi_arr     = np.array(ISI_VALUES, dtype=float)

# 3-D d' arrays: shape (n_s0, n_sig, n_eta)
results = {isi: grid[f'dprime_isi{isi}'] for isi in ISI_VALUES}

print(f'Grid: {len(sigma0_grid)} x {len(sigma_grid)} x {len(eta_grid)} '
      f'= {len(sigma0_grid)*len(sigma_grid)*len(eta_grid)} triples')
print(f'ISI values: {ISI_VALUES}')

# ── Exponential decay model and fitting helper ──
def exp_decay(isi, A, lam, C):
    """d'(ISI) = A * exp(-lam * ISI) + C"""
    return A * np.exp(-lam * isi) + C

FLAT_THRESH = 0.05  # std(d') below this → classify as flat

def fit_decay(dp_vals, isi_arr=isi_arr):
    """Fit exp_decay to 4 d' values.

    Returns dict with keys: A, lam, C, R2, curve_type.
    curve_type is one of 'exponential', 'flat', 'failed'.
    """
    dp = np.asarray(dp_vals, dtype=float)
    # Pre-classify flat curves
    if np.std(dp) < FLAT_THRESH:
        return dict(A=0.0, lam=0.0, C=np.mean(dp), R2=1.0,
                    curve_type='flat')
    try:
        p0 = (dp[0] - dp[-1], 0.1, dp[-1])
        popt, _ = curve_fit(exp_decay, isi_arr, dp, p0=p0,
                            bounds=([0, 0, -1], [15, 2, 15]),
                            maxfev=5000)
        A, lam, C = popt
        ss_res = np.sum((dp - exp_decay(isi_arr, *popt)) ** 2)
        ss_tot = np.sum((dp - np.mean(dp)) ** 2)
        R2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0
        return dict(A=A, lam=lam, C=C, R2=R2, curve_type='exponential')
    except Exception:
        return dict(A=np.nan, lam=np.nan, C=np.nan, R2=np.nan,
                    curve_type='failed')

# Median grid indices for fix-two-vary-one slicing
MED_S0  = len(sigma0_grid) // 2   # index 12
MED_SIG = len(sigma_grid)  // 2   # index 12
MED_ETA = len(eta_grid)    // 2   # index 10

param_info = [
    ('σ₀', sigma0_grid, 0),  # (label, grid_values, axis_in_3d)
    ('σ',  sigma_grid,  1),
    ('η',  eta_grid,    2),
]

## Fit Exponential Decay to All 12,500 Triples

For every (σ₀, σ, η) combination, extract the 4-point d' vs ISI curve and fit
$d'(\text{ISI}) = A\,e^{-\lambda\,\text{ISI}} + C$.  Curves with near-zero
variance (std < 0.05) are classified as **flat** without fitting.

In [ ]:
rows = []
for i_s0, s0 in enumerate(sigma0_grid):
    for i_sig, sig in enumerate(sigma_grid):
        for i_eta, eta in enumerate(eta_grid):
            dp_vals = np.array([results[isi][i_s0, i_sig, i_eta]
                                for isi in ISI_VALUES])
            fit = fit_decay(dp_vals)
            rows.append(dict(sigma0=s0, sigma=sig, eta=eta,
                             i_s0=i_s0, i_sig=i_sig, i_eta=i_eta,
                             **{f'dprime_isi{isi}': dp_vals[k]
                                for k, isi in enumerate(ISI_VALUES)},
                             **fit))

df = pd.DataFrame(rows)

# Reshape fit parameters into 3-D arrays for easy slicing
n_s0, n_sig, n_eta = len(sigma0_grid), len(sigma_grid), len(eta_grid)
A_3d   = df['A'].values.reshape(n_s0, n_sig, n_eta)
lam_3d = df['lam'].values.reshape(n_s0, n_sig, n_eta)
C_3d   = df['C'].values.reshape(n_s0, n_sig, n_eta)
R2_3d  = df['R2'].values.reshape(n_s0, n_sig, n_eta)

# Summary
print('=== Curve-type counts ===')
print(df['curve_type'].value_counts().to_string())

exp_mask = df['curve_type'] == 'exponential'
if exp_mask.any():
    print(f'\n=== R² for exponential fits (n={exp_mask.sum()}) ===')
    print(df.loc[exp_mask, 'R2'].describe().to_string())

print(f'\n=== Fit parameter ranges (exponential only) ===')
for col in ['A', 'lam', 'C']:
    vals = df.loc[exp_mask, col]
    print(f'  {col:>3s}: [{vals.min():.4f}, {vals.max():.4f}]  '
          f'median={vals.median():.4f}')

### Fit Quality Diagnostics

Distribution of R² values, curve-type breakdown, and example fits showing
data points with the fitted exponential overlay.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. R² histogram (exponential fits only)
ax = axes[0]
r2_vals = df.loc[exp_mask, 'R2']
ax.hist(r2_vals, bins=50, color='steelblue', edgecolor='white')
ax.axvline(0.8, color='red', linestyle='--', linewidth=1, label='R²=0.8')
ax.set_xlabel('R²')
ax.set_ylabel('Count')
ax.set_title('R² Distribution (exponential fits)')
ax.legend()

# 2. Curve-type bar chart
ax = axes[1]
counts = df['curve_type'].value_counts()
colors = {'exponential': 'steelblue', 'flat': '#2ca02c', 'failed': '#d62728'}
ax.bar(counts.index, counts.values,
       color=[colors.get(c, 'gray') for c in counts.index])
ax.set_ylabel('Count')
ax.set_title('Curve Classification')
for i, (ct, n) in enumerate(counts.items()):
    ax.text(i, n + 50, str(n), ha='center', fontsize=10)

# 3. A vs λ scatter coloured by R²
ax = axes[2]
exp_df = df[exp_mask]
sc = ax.scatter(exp_df['lam'], exp_df['A'], c=exp_df['R2'],
                cmap='viridis', s=8, alpha=0.5, vmin=0, vmax=1)
plt.colorbar(sc, ax=ax, label='R²')
ax.set_xlabel('λ (decay rate)')
ax.set_ylabel('A (amplitude)')
ax.set_title('A vs λ (coloured by R²)')

fig.tight_layout()
plt.show()

# ── Example fits: 3 good exponential, 3 flat, 3 low-R² ──
isi_fine = np.linspace(0, 16, 200)

good = exp_df.nlargest(3, 'R2')
flat_df = df[df['curve_type'] == 'flat'].sample(n=min(3, (df['curve_type']=='flat').sum()),
                                                 random_state=42)
marginal = exp_df[(exp_df['R2'] > 0.3) & (exp_df['R2'] < 0.85)].sample(
    n=min(3, len(exp_df)), random_state=42)
examples = pd.concat([good, flat_df, marginal]).reset_index(drop=True)
labels = (['Good exp.'] * len(good) + ['Flat'] * len(flat_df)
          + ['Marginal'] * len(marginal))

fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharey=False)
for idx, (ax, (_, row), lab) in enumerate(zip(axes.flat, examples.iterrows(), labels)):
    dp = [row[f'dprime_isi{isi}'] for isi in ISI_VALUES]
    ax.plot(isi_arr, dp, 'ko', markersize=7, label='data')
    if row['curve_type'] == 'exponential' and not np.isnan(row['A']):
        ax.plot(isi_fine, exp_decay(isi_fine, row['A'], row['lam'], row['C']),
                'r-', linewidth=1.5, label='fit')
    elif row['curve_type'] == 'flat':
        ax.axhline(row['C'], color='green', linestyle='--', label='flat')
    ax.set_title(f'{lab}  R²={row["R2"]:.3f}\n'
                 f'σ₀={row["sigma0"]:.3f} σ={row["sigma"]:.3f} η={row["eta"]:.3f}',
                 fontsize=9)
    ax.set_xlabel('ISI')
    ax.set_ylabel("d'")
    ax.legend(fontsize=7)

fig.suptitle('Example Fits', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## Fix-Two-Vary-One: d' vs ISI Curves

Fix two hyperparameters at their **median grid value** and sweep the third.
Each curve shows d' vs ISI for one setting of the varied parameter, coloured
by its value.  Smooth lines are the fitted exponential.

In [ ]:
isi_fine = np.linspace(0, 16, 200)

# For each panel: (varied_label, varied_grid, slice_func)
# slice_func(i) returns dp_vals array [4] for the i-th value of the varied param
slices = [
    ('σ₀ (encoding noise)', sigma0_grid,
     lambda i: np.array([results[isi][i, MED_SIG, MED_ETA] for isi in ISI_VALUES])),
    ('σ (diffusive noise)', sigma_grid,
     lambda i: np.array([results[isi][MED_S0, i, MED_ETA] for isi in ISI_VALUES])),
    ('η (drift step)', eta_grid,
     lambda i: np.array([results[isi][MED_S0, MED_SIG, i] for isi in ISI_VALUES])),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, pgrid, get_dp) in zip(axes, slices):
    norm = LogNorm(vmin=pgrid.min(), vmax=pgrid.max())
    cmap = cm.viridis

    for i, pval in enumerate(pgrid):
        dp = get_dp(i)
        color = cmap(norm(pval))
        ax.plot(isi_arr, dp, 'o', color=color, markersize=5)
        # Overlay fitted curve
        fit = fit_decay(dp)
        if fit['curve_type'] == 'exponential':
            ax.plot(isi_fine, exp_decay(isi_fine, fit['A'], fit['lam'], fit['C']),
                    '-', color=color, linewidth=1.2, alpha=0.8)
        else:
            ax.axhline(fit['C'], color=color, linewidth=0.8, alpha=0.4,
                       linestyle='--')

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=ax, label=label)
    ax.set_xlabel('ISI')
    ax.set_ylabel("d'")
    ax.set_title(f'Vary {label}')
    ax.set_xticks(isi_arr)

fig.suptitle(f'Fix-two-vary-one (fixed at median grid values: '
             f'σ₀={sigma0_grid[MED_S0]:.4f}, '
             f'σ={sigma_grid[MED_SIG]:.4f}, '
             f'η={eta_grid[MED_ETA]:.4f})',
             fontsize=11, y=1.02)
fig.tight_layout()
plt.show()

### Decay Parameters vs Each Hyperparameter

Extract fit parameters (A, λ, C) from the fix-two-vary-one slices and plot
them as functions of the varied hyperparameter.  Three slices per panel
(low / median / high for the fixed pair) show robustness of the trend.

In [ ]:
# Slice indices: low (25th pctile), med (50th), high (75th) of each grid
IDX_LO_S0, IDX_HI_S0  = n_s0  // 4, 3 * n_s0  // 4   # 6, 18
IDX_LO_SIG, IDX_HI_SIG = n_sig // 4, 3 * n_sig // 4   # 6, 18
IDX_LO_ETA, IDX_HI_ETA = n_eta // 4, 3 * n_eta // 4   # 5, 15

# For each varied param, define 3 slices of the fixed pair
# Each entry: (varied_label, varied_grid, list of (slice_label, A_1d, lam_1d, C_1d))
fit_param_names = ['A', 'lam', 'C']
fit_param_labels = ['A (amplitude)', 'λ (decay rate)', 'C (floor)']

def get_slice_1d(arr_3d, vary_axis, fix1_idx, fix2_idx):
    """Extract a 1-D slice along vary_axis with the other two fixed."""
    if vary_axis == 0:
        return arr_3d[:, fix1_idx, fix2_idx]
    elif vary_axis == 1:
        return arr_3d[fix1_idx, :, fix2_idx]
    else:
        return arr_3d[fix1_idx, fix2_idx, :]

# Define the 3 panels (rows): vary σ₀, vary σ, vary η
vary_specs = [
    ('σ₀', sigma0_grid, 0,
     [(IDX_LO_SIG, IDX_LO_ETA, 'low σ,η'),
      (MED_SIG, MED_ETA, 'med σ,η'),
      (IDX_HI_SIG, IDX_HI_ETA, 'high σ,η')]),
    ('σ', sigma_grid, 1,
     [(IDX_LO_S0, IDX_LO_ETA, 'low σ₀,η'),
      (MED_S0, MED_ETA, 'med σ₀,η'),
      (IDX_HI_S0, IDX_HI_ETA, 'high σ₀,η')]),
    ('η', eta_grid, 2,
     [(IDX_LO_S0, IDX_LO_SIG, 'low σ₀,σ'),
      (MED_S0, MED_SIG, 'med σ₀,σ'),
      (IDX_HI_S0, IDX_HI_SIG, 'high σ₀,σ')]),
]

line_styles = ['-', '--', ':']
fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for row_idx, (vary_label, pgrid, vary_axis, fix_list) in enumerate(vary_specs):
    for col_idx, (fp_arr, fp_label) in enumerate(
            zip([A_3d, lam_3d, C_3d], fit_param_labels)):
        ax = axes[row_idx, col_idx]
        for (fix1, fix2, slabel), ls in zip(fix_list, line_styles):
            vals = get_slice_1d(fp_arr, vary_axis, fix1, fix2)
            ax.plot(pgrid, vals, ls, linewidth=2, label=slabel)
        ax.set_xscale('log')
        ax.set_xlabel(vary_label)
        if col_idx == 0:
            ax.set_ylabel(f'Vary {vary_label}', fontsize=11)
        if row_idx == 0:
            ax.set_title(fp_label, fontsize=11)
        ax.legend(fontsize=7)

fig.suptitle('Fit Parameters vs Each Hyperparameter (3 slices of fixed pair)',
             fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 2D Heatmaps of Decay Parameters

Each row fixes one hyperparameter at its median and shows A, λ, C as a
heatmap over the remaining two.  Reveals pairwise interaction effects.

In [ ]:
# Heatmap slices: (row_label, 2d_slice_func, y_grid, x_grid, ylabel, xlabel)
heatmap_specs = [
    ('Fix σ₀ at median',
     lambda arr: arr[MED_S0, :, :],   # shape (n_sig, n_eta)
     sigma_grid, eta_grid, 'σ', 'η'),
    ('Fix σ at median',
     lambda arr: arr[:, MED_SIG, :],  # shape (n_s0, n_eta)
     sigma0_grid, eta_grid, 'σ₀', 'η'),
    ('Fix η at median',
     lambda arr: arr[:, :, MED_ETA],  # shape (n_s0, n_sig)
     sigma0_grid, sigma_grid, 'σ₀', 'σ'),
]

fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for row_idx, (row_label, slicer, ygrid, xgrid, ylabel, xlabel) in enumerate(heatmap_specs):
    for col_idx, (arr_3d, fp_label) in enumerate(
            zip([A_3d, lam_3d, C_3d], fit_param_labels)):
        ax = axes[row_idx, col_idx]
        data_2d = slicer(arr_3d)
        im = ax.pcolormesh(xgrid, ygrid, data_2d, shading='nearest',
                           cmap='magma')
        plt.colorbar(im, ax=ax)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel(xlabel)
        if col_idx == 0:
            ax.set_ylabel(f'{row_label}\n{ylabel}', fontsize=10)
        if row_idx == 0:
            ax.set_title(fp_label, fontsize=11)

fig.suptitle('2D Heatmaps of Decay Parameters (fixing one param at median)',
             fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## Non-Exponential Regime: Where Curves Are Flat

Triples classified as **flat** (std(d') < 0.05) indicate parameter regimes
where memory does not degrade over the ISI range — either because noise is
too low to cause decay, or d' is already at floor.  Visualising where these
occur in parameter space reveals the boundary between decaying and
non-decaying regimes.

In [ ]:
is_flat_3d = (df['curve_type'] == 'flat').values.reshape(n_s0, n_sig, n_eta)

# Fraction-flat heatmaps: average over the fixed axis
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

flat_specs = [
    ('Fix σ₀ at median', is_flat_3d[MED_S0, :, :].astype(float),
     sigma_grid, eta_grid, 'σ', 'η'),
    ('Fix σ at median', is_flat_3d[:, MED_SIG, :].astype(float),
     sigma0_grid, eta_grid, 'σ₀', 'η'),
    ('Fix η at median', is_flat_3d[:, :, MED_ETA].astype(float),
     sigma0_grid, sigma_grid, 'σ₀', 'σ'),
]

for ax, (title, frac, ygrid, xgrid, ylabel, xlabel) in zip(axes, flat_specs):
    im = ax.pcolormesh(xgrid, ygrid, frac, shading='nearest',
                       cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='Flat fraction')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

fig.suptitle('Where Are Curves Flat? (green = flat / no decay, red = exponential decay)',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

# Also show marginal flat fraction vs each parameter
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (label, pgrid, axis) in zip(axes, param_info):
    # Average flat fraction over the other two axes
    frac = is_flat_3d.mean(axis=tuple(i for i in range(3) if i != axis))
    ax.plot(pgrid, frac, 'o-', color='steelblue', linewidth=2)
    ax.set_xscale('log')
    ax.set_xlabel(label)
    ax.set_ylabel('Fraction flat')
    ax.set_title(f'Flat fraction vs {label}')
    ax.set_ylim(-0.05, 1.05)

fig.tight_layout()
plt.show()

## Summary: How Each Hyperparameter Controls Decay Shape

Spearman rank correlations between each hyperparameter and each fit
parameter quantify the monotonic relationships.  A bar chart makes the
dominant drivers visible at a glance.

In [ ]:
# Use only exponential fits for correlation analysis
edf = df[df['curve_type'] == 'exponential'].copy()

hyper_names = ['sigma0', 'sigma', 'eta']
hyper_labels = ['σ₀', 'σ', 'η']
fit_names = ['A', 'lam', 'C']

# Spearman correlation table
corr_table = pd.DataFrame(index=hyper_labels, columns=fit_param_labels)
pval_table = pd.DataFrame(index=hyper_labels, columns=fit_param_labels)

for hi, (hname, hlabel) in enumerate(zip(hyper_names, hyper_labels)):
    for fi, (fname, flabel) in enumerate(zip(fit_names, fit_param_labels)):
        rho, pval = spearmanr(edf[hname], edf[fname])
        corr_table.loc[hlabel, flabel] = rho
        pval_table.loc[hlabel, flabel] = pval

print('=== Spearman Rank Correlations (exponential fits only) ===')
print(corr_table.to_string(float_format=lambda x: f'{x:.3f}'))
print(f'\n(n = {len(edf)} exponential fits)')

# Bar chart of absolute correlations
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
x = np.arange(len(hyper_labels))
width = 0.6
colors_bar = ['#1f77b4', '#ff7f0e', '#2ca02c']

for col_idx, fp_label in enumerate(fit_param_labels):
    ax = axes[col_idx]
    vals = corr_table[fp_label].astype(float)
    bar_colors = ['#d62728' if v < 0 else '#2ca02c' for v in vals]
    ax.bar(x, vals, width, color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(hyper_labels, fontsize=12)
    ax.set_title(fp_label, fontsize=11)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylim(-1, 1)
    if col_idx == 0:
        ax.set_ylabel('Spearman ρ')

fig.suptitle('Hyperparameter → Decay Parameter Correlations', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

# Text summary
print('\n=== Interpretation ===')
for flabel in fit_param_labels:
    vals = corr_table[flabel].astype(float)
    dominant = vals.abs().idxmax()
    direction = 'increases' if vals[dominant] > 0 else 'decreases'
    print(f'  {flabel} is most strongly driven by {dominant} '
          f'(ρ = {vals[dominant]:.3f}, {direction} with {dominant})')